In [1]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))
from constants import *

In [2]:
pairwise_alignment = pd.read_csv(
    "../data/swissprot/2024_01/diamond_swissprot_2024_01_alignment.tsv",
    sep="\t",
    header=None,
    names=[
        "query_id",
        "subject_id",
        "perc_identity",
        "align_length",
        "mismatches",
        "gap_opens",
        "q_start",
        "q_end",
        "s_start",
        "s_end",
        "e_value",
        "bit_score",
    ],
)
pairwise_alignment = pairwise_alignment[pairwise_alignment["query_id"] != pairwise_alignment["subject_id"]]
pairwise_alignment

,query_id,subject_id,perc_identity,align_length,mismatches,gap_opens,q_start,q_end,s_start,s_end,e_value,bit_score
1,Q6GZX4,Q91FP2,33.7,172,100,3,96,253,208,379,3.720000e-25,105.0
2,Q6GZX4,Q196Y1,29.5,210,124,5,65,254,186,391,2.060000e-19,89.7
4,Q6GZX3,Q197B3,34.3,283,156,12,2,264,3,275,3.340000e-43,156.0
5,Q6GZX3,Q91FI7,32.2,270,144,8,27,264,88,350,7.770000e-41,150.0
12,Q6GZX0,P14362,33.6,122,79,2,79,199,14,134,7.780000e-13,66.2
...,...,...,...,...,...,...,...,...,...,...,...,...
12552650,B2ZDY1,O73557,44.2,95,46,2,1,95,1,88,2.070000e-19,78.6
12552651,B2ZDY1,Q27YE2,40.2,92,55,0,1,92,1,92,8.820000e-19,77.0
12552652,B2ZDY1,P18541,44.9,69,36,2,28,94,21,89,2.320000e-14,65.5
12552653,B2ZDY1,P19325,40.0,75,37,2,1,74,1,68,1.810000e-10,55.1


In [3]:
# Get all unique proteins from alignment (uses accession IDs, e.g. Q6GZX4)
alignment_proteins = set(pairwise_alignment['query_id'].unique()) | set(pairwise_alignment['subject_id'].unique())

# Load swissprot annotations to find unaligned proteins.
swissprot_all = pd.read_csv(
    "../data/swissprot/2024_01/swissprot_2024_01_annotations.tsv",
    sep="\t",
    usecols=["Entry Name"],
).rename(columns={"Entry Name": "EntryID"})

unaligned_proteins = set(swissprot_all["EntryID"].dropna().unique()) - alignment_proteins
all_proteins = alignment_proteins | unaligned_proteins
print(f"Alignment proteins : {len(alignment_proteins)}")
print(f"Unaligned proteins : {len(unaligned_proteins)}")
print(f"Total proteins     : {len(all_proteins)}")

Alignment proteins : 553160
Unaligned proteins : 17589
Total proteins     : 570749


In [4]:
# Select df with perc_identity >= 30 %
h30 = pairwise_alignment[pairwise_alignment['perc_identity'] > 30]
proteins_h30 = all_proteins - set(h30['subject_id'].unique()) - set(h30['query_id'].unique())
len(proteins_h30)

21424

In [36]:
os.makedirs("../data/H30", exist_ok=True)

for ontology in SUBONTOLOGIES:
    input_path = f"../data/swissprot/2024_01/swissprot_2024_01_{ontology}_exp_annotations.tsv"
    output_path = f"../data/H30/H30_{ontology}_test_annotations.tsv"
    df = pd.read_csv(input_path, sep="\t")
    filtered_df = df[df['EntryID'].isin(proteins_h30)]
    filtered_df.to_csv(output_path, sep="\t", index=False)

In [5]:
# Build Entry Name (D1 format: e.g. 140U_DROME) → accession (H30 format: e.g. A0A060A682) mapping
entryname_to_acc = pd.read_csv(
    "../data/swissprot/2024_01/swissprot_2024_01_annotations.tsv",
    sep="\t",
    usecols=["EntryID", "Entry Name"],
).drop_duplicates().set_index("EntryID")["Entry Name"].to_dict()

# Collect all H30 test proteins (accession format) across all ontologies to avoid leakage
h30_test_proteins = set()
for ontology in SUBONTOLOGIES:
    df = pd.read_csv(f"../data/H30/H30_{ontology}_test_annotations.tsv", sep="\t")
    h30_test_proteins |= set(df["EntryID"].dropna().unique())
print(f"H30 test proteins to exclude: {len(h30_test_proteins)}")

# Build H30 train files from D1 train, remapping IDs and removing test proteins
for ontology in SUBONTOLOGIES:
    d1_train = pd.read_csv(f"../data/D1/D1_{ontology}_train_annotations.tsv", sep="\t")
    # D1 uses Entry Name format; remap to accession to match H30
    d1_train["EntryID"] = d1_train["EntryID"].map(entryname_to_acc)
    n_unmapped = d1_train["EntryID"].isna().sum()
    d1_train = d1_train.dropna(subset=["EntryID"])
    # Remove any H30 test protein to prevent leakage
    d1_train = d1_train[~d1_train["EntryID"].isin(h30_test_proteins)]
    d1_train.to_csv(f"../data/H30/H30_{ontology}_train_annotations.tsv", sep="\t", index=False)
    print(f"H30 {ontology} train: {d1_train['EntryID'].nunique()} proteins  "
          f"(unmapped D1 IDs dropped: {n_unmapped})")

H30 test proteins to exclude: 3900
H30 BPO train: 45453 proteins  (unmapped D1 IDs dropped: 320)
H30 MFO train: 30686 proteins  (unmapped D1 IDs dropped: 197)
H30 CCO train: 40729 proteins  (unmapped D1 IDs dropped: 305)


In [39]:
import subprocess
import sys

# Train set = full swissprot 2024_01 experimental annotations.
# background.py internally excludes H30 test proteins from IC calculation.
cmd = [
    sys.executable, "../background.py",
    "--bpo",      "../data/swissprot/2024_01/swissprot_2024_01_BPO_exp_annotations.tsv",
    "--cco",      "../data/swissprot/2024_01/swissprot_2024_01_CCO_exp_annotations.tsv",
    "--mfo",      "../data/swissprot/2024_01/swissprot_2024_01_MFO_exp_annotations.tsv",
    "--test_bpo", "../data/H30/H30_BPO_test_annotations.tsv",
    "--test_cco", "../data/H30/H30_CCO_test_annotations.tsv",
    "--test_mfo", "../data/H30/H30_MFO_test_annotations.tsv",
    "--output",   "../data/H30/background_H30.pkl",
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("Background file written to ../data/H30/background_H30.pkl")

Loading CCO file: ../data/swissprot/2024_01/swissprot_2024_01_CCO_exp_annotations.tsv
Loading BPO file: ../data/swissprot/2024_01/swissprot_2024_01_BPO_exp_annotations.tsv
Loading MFO file: ../data/swissprot/2024_01/swissprot_2024_01_MFO_exp_annotations.tsv
Merged background file saved to: ../data/H30/background_H30.pkl

Background file written to ../data/H30/background_H30.pkl


In [ ]:
swissprot_ids = set(swissprot_all["EntryID"].dropna().unique())
still_missing = swissprot_ids - proteins_h30 - set(h30['query_id'].unique()) - set(h30['subject_id'].unique())
print(f"Proteins in swissprot annotations : {len(swissprot_ids)}")
print(f"Proteins kept in H30 (no >30% hit): {len(proteins_h30)}")
print(f"Proteins excluded (have >30% hit) : {len(swissprot_ids - proteins_h30)}")
print(f"Still missing after fix            : {len(still_missing)}")  # should be 0

# H30 proteins split by origin: unaligned (no DIAMOND hit) vs low-identity-aligned (hits but all ≤30%)
h30_unaligned = proteins_h30 & unaligned_proteins
h30_aligned   = proteins_h30 & alignment_proteins
print(f"\nOverall H30 breakdown:")
print(f"  Unaligned (no DIAMOND hit)  : {len(h30_unaligned)} ({100*len(h30_unaligned)/len(proteins_h30):.1f}%)")
print(f"  Low-identity aligned (≤30%) : {len(h30_aligned)}  ({100*len(h30_aligned)/len(proteins_h30):.1f}%)")

# Proteins dropped from every output file due to having no experimental annotations
annotated_in_any = set()
for ontology in SUBONTOLOGIES:
    input_path = f"../data/swissprot/2024_01/swissprot_2024_01_{ontology}_exp_annotations.tsv"
    ann_df = pd.read_csv(input_path, sep="\t", usecols=["EntryID"])
    annotated_in_any |= set(ann_df["EntryID"].dropna().unique())

no_annotation = proteins_h30 - annotated_in_any
print(f"\nH30 proteins with no experimental annotation (any ontology): "
      f"{len(no_annotation)} / {len(proteins_h30)} ({100*len(no_annotation)/len(proteins_h30):.1f}%)")

# stats
print()
for ontology in SUBONTOLOGIES:
    output_path = f"../data/H30/H30_{ontology}_test_annotations.tsv"
    df = pd.read_csv(output_path, sep="\t")
    onto_proteins = set(df["EntryID"].unique())
    n_unaligned = len(onto_proteins & h30_unaligned)
    n_aligned   = len(onto_proteins & h30_aligned)
    total       = len(onto_proteins)
    n_dropped   = len(proteins_h30 - onto_proteins)
    print(f"H30 {ontology}: {total} proteins total  |  "
          f"unaligned: {n_unaligned} ({100*n_unaligned/total:.1f}%)  |  "
          f"low-identity aligned: {n_aligned} ({100*n_aligned/total:.1f}%)  |  "
          f"dropped (no annotation): {n_dropped} ({100*n_dropped/len(proteins_h30):.1f}%)")

Proteins in swissprot annotations : 570749
Proteins kept in H30 (no >30% hit): 21424
Proteins excluded (have >30% hit) : 549325
Still missing after fix            : 0

Overall H30 breakdown:
  Unaligned (no DIAMOND hit)  : 17589 (82.1%)
  Low-identity aligned (≤30%) : 3835  (17.9%)

H30 proteins with no experimental annotation (any ontology): 17524 / 21424 (81.8%)

H30 BPO: 3091 proteins total  |  unaligned: 1990 (64.4%)  |  low-identity aligned: 1101 (35.6%)  |  dropped (no annotation): 18333 (85.6%)
H30 MFO: 1545 proteins total  |  unaligned: 925 (59.9%)  |  low-identity aligned: 620 (40.1%)  |  dropped (no annotation): 19879 (92.8%)
H30 CCO: 2757 proteins total  |  unaligned: 1797 (65.2%)  |  low-identity aligned: 960 (34.8%)  |  dropped (no annotation): 18667 (87.1%)
